<div style="background-color: #0f3460; background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 30px 20px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #ffffff; margin: 0; font-size: 2em;">🤖 Lab 02: Create Your First Prompt Agent</h1>
  <p style="color: #cccccc; font-size: 1.1em; margin-top: 8px;">
    Contoso Travel Workshop — Microsoft Foundry Agent Observability
  </p>
</div>

## 🎯 ここで学ぶこと

このラボでは、Azure AI Projects SDK を使用して最初の **プロンプトエージェント** — **Contoso Travel コンシェルジュ** を作成します。カスタム指示を持つエージェントの定義、会話の開始、メッセージの送信、マルチターンのやり取りの処理方法を学びます。コンシェルジュは、顧客を迎え、一般的な旅行の質問に答えるフロントデスクエージェントです。後のラボでは、専門的なツールで強化し、ドメイン固有のエージェントに接続します。

---

## 🧳 Contoso Travel のストーリー

**Contoso Travel** には AI コンシェルジュが必要です — 顧客を迎え、どのような旅行を求めているかを理解し、計画の過程を案内するフレンドリーなフロントデスクエージェントです。これまでに出会った中で最高の旅行アドバイザーを思い浮かべてください: 温かく、知識豊富で、夢の休暇について自然な会話ができる人です。**環境はすでに整っています**（ラボ 00–01）。SDK はインストール済みで、資格情報は検証済み、サンプルデータも読み込まれています。さて、実際のエージェントを作成する時です。

### 🔴 現在直面している問題
- Foundry SDK を使って AI エージェントを構築したことがない。
- エージェントに個性を与え、指示を通じて行動を定義し、会話を通じてやり取りする必要があることは分かっている — しかし **実際にどうやって作成するのか？**
- 指示を定義する正しい方法は？会話はどのように機能するのか？顧客がリクエストを修正するマルチターンのやり取りはどう処理するのか？
- 作業が終わったらリソースをどのようにクリーンアップするのか？

### ✅ このラボで解決すること

このラボでは、ゼロから動作する **プロンプトエージェント** — Contoso Travel コンシェルジュ を作成します:
 - `PromptAgentDefinition` を使ってエージェントを定義する、
 - 会話を作成し、メッセージを送信する、
 - マルチターンの対話を処理する、
 - レスポンスオブジェクトを確認する。

これは、他のすべてのラボの基礎となる部分です。

### 🧠 メンタルモデル：エージェントをチームメンバーとして捉える

AIエージェントの構築は、Contoso Travelに新しいチームメンバーを採用するようなものだと考えてください:

| 現実世界 | エージェントの概念 | SDK コンポーネント |
|------------|---------------|---------------|
| 職務記述書を書く | エージェントの指示を定義する | `PromptAgentDefinition` |
| 役割を割り当てる | エージェントに名前を付ける | `agent_name` パラメータ |
| 会議を開始する | 会話を開始する | `conversations.create()` |
| 質問をする | メッセージを送信する | `responses.create()` |
| 議論を続ける | マルチターンのフォローアップ | 同じ `conversation.id` |
| 会議を終了する | 会話を削除する | `conversations.delete()` |

**コンシェルジュ**はあなたの最初の採用です — フレンドリーなフロントデスクエージェントです。後のラボでは、専門家（フライト、ホテル、車）やマネージャー（オーケストレーター）を採用して調整します。

## 1. セットアップ

環境変数を読み込み、Azure AI プロジェクトクライアントを作成します。

---


In [ ]:
# Setup: load credentials and create SDK clients
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
# PromptAgentDefinition: declarative config for an
# LLM agent (model + instructions + optional tools)
from azure.ai.projects.models import PromptAgentDefinition

# .env lives one level above the notebooks dir
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))

endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_name = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]

credential = DefaultAzureCredential()
# AIProjectClient manages agents; openai_client handles
# conversations and responses via OpenAI-compatible API
project_client = AIProjectClient(endpoint=endpoint, credential=credential)
openai_client = project_client.get_openai_client()

print(f"✅ Connected to Microsoft Foundry")
print(f"   Model: {model_name}")

## 2. コンシェルジュエージェントを作成する

旅行に特化した指示でエージェントを定義します。システムプロンプトは、エージェントが顧客の質問にどのように応答するかを形作ります。

---


In [ ]:
# システムプロンプトを定義 — エージェントの
# 性格、境界、応答スタイルを形作る
CONCIERGE_INSTRUCTIONS = """あなたは Contoso Travel のコンシェルジュであり、フレンドリーで知識豊富な旅行アシスタントです。

あなたの責務:
- 目的地、旅行のヒント、物流に関する質問に答えることで、顧客の旅行計画を支援する
- 有益で正確かつ簡潔な旅行アドバイスを提供する
- 応答は温かく、プロフェッショナルであること
- 特定のデータがない場合は、一般的な旅行ガイドラインを提供する
- Contoso Travel がフライト、ホテル、レンタカーの手配を支援できることを常に言及する

Remember: あなたは高級旅行代理店であるコントソ・トラベルの代表者です。
"""

# create_version は、エージェントの不変のスナップショットを作成します。
# 呼び出しごとに新しいバージョンが生成されます。これは、
# A/B テスト、ロールバック、および監査証跡に役立ちます。
agent = project_client.agents.create_version(
    agent_name="contoso-travel-concierge",
    definition=PromptAgentDefinition(
        model=model_name,
        instructions=CONCIERGE_INSTRUCTIONS,
    ),
)

print(f"✅ Agent created!")
print(f"   Name: {agent.name}")
print(f"   Version: {agent.version}")
print(f"   ID: {agent.id}")

## 3. 会話を開始する

会話は複数のメッセージにわたってコンテキストを維持します。ここでは、会話を作成し、最初の旅行に関する問い合わせを送信します。

---


In [ ]:
# 会話を作成 — サーバー側のセッションで
# マルチターンのコンテキストを追跡
conversation = openai_client.conversations.create()
print(f"✅ 会話が作成されました (ID: {conversation.id})")

# extra_body の agent_reference は、この応答を特定の名前付きエージェントにバインドします（最新バージョンを使用）
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    input="こんにちは！パリへの旅行を計画しています。何を知っておくべきですか？",
)

# output_text: response.output アイテムからすべてのテキストコンテンツを連結する便利なアクセサー
print(f"\n🤖 コンシェルジュの応答:\n")
print(response.output_text)

## 4. マルチターンの会話

エージェントは会話内でコンテキストを維持します。フォローアップの質問をして、マルチターンの対話をどのように処理するかを確認しましょう。

---


In [ ]:
# 同じ conversation.id = エージェントは完全なコンテキストを保持します。
# パリについて尋ねていることを再度述べる必要はありません。
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    input="一年の中で訪れるのに最適な時期はいつですか？",
)

print(f"🤖 コンシェルジュの応答:\n")
print(response.output_text)

In [ ]:
# 三回目のターン — エージェントは依然としてパリの会話全体を保持しています
# サーバー側のメッセージストアからの履歴
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    input="一週間の旅行の予算についてアドバイスをいただけますか？",
)

print(f"🤖 コンシェルジュの応答:\n")
print(response.output_text)

## 5. レスポンスオブジェクトの探索

レスポンスの構造を確認して、どのデータが利用可能かを理解しましょう。

---


In [ ]:
# レスポンスオブジェクトの構造を確認します。
# output は型付きアイテムのリストです（例: 'message'）
# テキストのようなコンテンツブロックを含むアイテム。
print(f"レスポンス ID: {response.id}")
print(f"モデル: {response.model}")
print(f"ステータス: {response.status}")
print(f"出力テキスト: {response.output_text[:100]}...")
print(f"\n出力アイテム ({len(response.output)} 件):")
for i, item in enumerate(response.output):
    print(f"  [{i}] Type: {item.type}")
    if hasattr(item, 'content'):
        for j, content in enumerate(item.content):
            print(f"       Content[{j}]: {type(content).__name__}")

# トークン使用量 — コスト監視のために追跡
print(f"\n使用量:")
print(f"  入力トークン:  {response.usage.input_tokens}")
print(f"  出力トークン: {response.usage.output_tokens}")
print(f"  合計トークン:  {response.usage.total_tokens}")

## 6. クリーンアップ

作業が完了したら、リソースをクリーンアップしましょう — 会話とエージェントバージョンを削除します。

---


In [ ]:
# クリーンアップ: サーバー側のリソースを解放します。
# 会話は明示的に削除されるまで保持されます。
openai_client.conversations.delete(conversation_id=conversation.id)
print("✅ Conversation deleted")

# Deletes this version only; other versions (if any)
# of the same agent name remain available.
project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
print("✅ Agent version deleted")

# Release HTTP connections and token caches
openai_client.close()
project_client.close()
credential.close()
print("✅ Clients closed")

## 7. Summary & Next Steps

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #2e8b57; padding: 18px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <h3 style="margin-top: 0; color: #1a1a1a;">✅ What We Accomplished</h3>
  <ul style="margin-bottom: 0;">
  <li>カスタム旅行コンシェルジュ指示付きのプロンプトエージェントを作成しました</li>
  <li>マルチターンのコンテキスト保持を実証しました</li>
  <li>レスポンスオブジェクトの構造を探索しました</li>
  </ul>
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #1565c0; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <strong>💡 Key Takeaway:</strong> Contoso Travelのコンシェルジュは一般的な質問には答えられますが、実際のフライト、ホテル、レンタカーのデータにはアクセスできません。回答はトレーニングデータに基づいています。
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #ff9800; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <strong>➡️ Next:</strong> <a href="lab-03a-tools.ipynb">Lab 03a</a>では、<b>Function Tools</b>を追加して、エージェントがContoso TravelのCSVデータにアクセスできるようにします — これにより、真のデータ駆動型旅行アシスタントになります！
</div>

---
